<a href="https://colab.research.google.com/github/djmcnay/minty-box/blob/master/openwakeword_model_training_simple_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train a custom openWakeWord model

Runs end-to-end on Google Colab. Workflow:

1. **Set params** in the next cell (just `WAKE_WORD` if you want to keep defaults).
2. **Run the setup cell.** It clones repos, installs deps with `uv` (fast), then auto-restarts the kernel so a clean numpy/pyarrow ABI is loaded.
3. **After the restart**, choose `Runtime → Run all` (or just run the remaining cells top-to-bottom). Setup is idempotent and will skip on the second pass.
4. The wake-word preview cell lets you sanity-check pronunciation — if it sounds wrong, try a phonetic spelling (e.g. `hey siri` → `hey_seer_e`, `2` → `two`, no punctuation except `?` / `!`).
5. Total runtime: ~10 min download + 30–60 min training on a CPU runtime (much faster on a GPU).
6. **Output:** the trained `.onnx` and `.tflite` land alongside the notebook as `<wake_word>.{onnx,tflite}` (spaces replaced with `_`). Move them into `models/` afterwards. In Colab they also auto-download via the browser.

The data used here is **non-commercial / personal use only** (mixed licenses).

*Optional:* set an `HF_TOKEN` in Colab Secrets (key icon in the left sidebar) for higher Hugging Face rate limits. Anonymous downloads work but are throttled. Get a token at https://huggingface.co/settings/tokens (role = Read).

In [1]:
# ============================================================
# USER PARAMETERS — edit these
# ============================================================
WAKE_WORD = "Araminta"            # The phrase to detect
N_SAMPLES = 1000                  # TTS examples to generate (1000 is fast; 30k–50k is best)
N_TRAINING_STEPS = 10000          # 10k usually works; longer is usually better
FALSE_ACTIVATION_PENALTY = 1500   # Higher = fewer false positives (but also fewer true positives)
N_HOURS_BACKGROUND = 1            # Hours of background audio to stream from each of AudioSet + FMA

## 1. Setup: clone repos, install deps, restart kernel

The kernel restart at the end is required: several pinned packages need a numpy<2 ABI, but Colab's pre-installed pandas is built against numpy 2.x. A clean restart loads everything against the same ABI and prevents the `numpy.dtype size changed` / `pyarrow.PyExtensionType` import errors.

This cell is idempotent — after the restart it will short-circuit on the second run. The marker file is version-stamped, so if pins change later, bumping `SETUP_VERSION` re-triggers the install automatically.

In [2]:
import os, sys

# Bump SETUP_VERSION when install pins change so re-running this cell
# automatically reinstalls (no need to manually clear the marker).
SETUP_VERSION = "3"
MARKER = f"/tmp/.minty_setup_done_v{SETUP_VERSION}"

if os.path.exists(MARKER):
    print("Setup already complete — skipping.")
else:
    # --- Clone source repos -------------------------------------------------
    if not os.path.exists("./piper-sample-generator"):
        !git clone -q https://github.com/rhasspy/piper-sample-generator
        !wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
        !cd piper-sample-generator && git checkout -q 213d4d5

    if not os.path.exists("./openwakeword"):
        !git clone -q https://github.com/dscripka/openwakeword

    # --- Install deps via uv (much faster than pip) -------------------------
    !pip install -q uv

    # Torch wheels live on a separate index
    !uv pip install --system -q \
        torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 \
        --index-url https://download.pytorch.org/whl/cu121

    # Version pin notes:
    #  - numpy<2 / pandas<2.2: keep numpy/pandas on the pre-numpy-2 ABI so the
    #    older pinned training deps (datasets 2.14.6, audiomentations 0.33, etc.)
    #    don't blow up at import time with `numpy.dtype size changed`.
    #  - pyarrow<15: pyarrow 15 removed `PyExtensionType`, which datasets 2.14.6
    #    still uses; AttributeError at `import datasets` without this pin.
    #  - tensorflow: required at runtime by onnx2tf for the ONNX→TFLite step.
    #    Without it onnx2tf errors silently and no .tflite is produced.
    !uv pip install --system -q \
        "numpy<2" "pandas<2.2" "pyarrow<15" \
        tensorflow \
        piper-tts piper-phonemize-cross webrtcvad \
        mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
        speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
        acoustics==0.2.6 onnxruntime==1.22.1 ai_edge_litert==1.4.0 \
        onnxsim onnx2tf "onnx==1.19.1" onnx_graphsurgeon sng4onnx \
        pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19 \
        scipy tqdm pyyaml

    !uv pip install --system -q -e ./openwakeword --no-deps

    # --- Bundled openWakeWord assets (not on PyPI) --------------------------
    models_dir = "openwakeword/openwakeword/resources/models"
    os.makedirs(models_dir, exist_ok=True)
    base = "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1"
    for fname in ("embedding_model.onnx", "embedding_model.tflite",
                  "melspectrogram.onnx", "melspectrogram.tflite"):
        dest = f"{models_dir}/{fname}"
        if not os.path.exists(dest):
            !wget -q -O {dest} {base}/{fname}

    # Clean up any stale older-version markers
    !rm -f /tmp/.minty_setup_done /tmp/.minty_setup_done_v1 /tmp/.minty_setup_done_v2
    open(MARKER, "w").close()
    print("Setup complete. Restarting kernel so the new numpy/pyarrow ABI is loaded...")
    os.kill(os.getpid(), 9)

Setup already complete — skipping.


## 2. Imports and helpers

In [3]:
import os
import sys
import shutil
import locale
from pathlib import Path

import numpy as np
import scipy.io.wavfile
import yaml
import datasets
from tqdm import tqdm
from IPython.display import Audio

for p in ("piper-sample-generator", "openwakeword"):
    if p not in sys.path:
        sys.path.append(p)

from generate_samples import generate_samples

# Colab's locale lookup can return None and break downstream HF code
locale.getpreferredencoding = lambda do_setlocale=True: "UTF-8"


def synthesize(text, out_path="test_generation.wav"):
    """Generate a single TTS sample of `text` using piper."""
    generate_samples(
        text=text, max_samples=1,
        length_scales=[1.1], noise_scales=[0.7], noise_scale_ws=[0.7],
        output_dir="./", batch_size=1, auto_reduce_batch_size=True,
        file_names=[out_path],
    )
    return out_path


def to_16khz_wav(src_paths, out_dir):
    """Resample a collection of local audio files to 16kHz, write as 16-bit PCM WAV."""
    os.makedirs(out_dir, exist_ok=True)
    ds = datasets.Dataset.from_dict({"audio": [str(p) for p in src_paths]})
    ds = ds.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds):
        stem = Path(row["audio"]["path"]).stem
        scipy.io.wavfile.write(
            f"{out_dir}/{stem}.wav", 16000,
            (row["audio"]["array"] * 32767).astype(np.int16),
        )


def stream_to_wav(hf_iter, out_dir, n_clips):
    """Stream rows from a HF audio dataset iterator, write 16-bit 16kHz WAVs.
    Stops early if the iterator is exhausted before n_clips."""
    os.makedirs(out_dir, exist_ok=True)
    written = 0
    for _ in tqdm(range(n_clips)):
        try:
            row = next(hf_iter)
        except StopIteration:
            break
        src_path = row["audio"].get("path")
        stem = Path(src_path).stem if src_path else f"clip_{written:06d}"
        scipy.io.wavfile.write(
            f"{out_dir}/{stem}.wav", 16000,
            (row["audio"]["array"] * 32767).astype(np.int16),
        )
        written += 1
    return written

## 3. Preview the wake word

Make sure the generated TTS sounds right. If not, edit `WAKE_WORD` above (try phonetic spellings) and re-run this cell.

In [4]:
synthesize(WAKE_WORD)
Audio("test_generation.wav", autoplay=True)

/content/piper-sample-generator/generate_samples.py:76: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch_model = torch.load(model_path)


## 4. Download training data

~10 minutes. Pulls room impulse responses (echo), AudioSet noise, FMA music, and pre-computed openWakeWord features. Each block is guarded so it's safe to re-run.

AudioSet and FMA stream from Hugging Face. AudioSet sometimes fails to load with the pinned older `datasets` library (schema parser quirks); the load is wrapped in try/except so a failure just logs a warning and training proceeds with FMA-only background. Set `HF_TOKEN` in Colab Secrets for faster downloads.

In [5]:
# Room Impulse Responses (~2 min) — adds realistic echo during augmentation
if not os.path.exists("./mit_rirs"):
    !git lfs install
    !git clone -q https://huggingface.co/datasets/davidscripka/MIT_environmental_impulse_responses
    to_16khz_wav(Path("MIT_environmental_impulse_responses/16khz").glob("*.wav"), "./mit_rirs")

# AudioSet — environmental noise / speech / etc. (10-second clips, CC-BY-4.0, ungated).
# Wrapped in try/except: the pinned datasets==2.14.6 can fail to parse newer schemas
# (e.g. TypeError from features.py). If it fails, training falls back to FMA-only.
if not os.path.exists("./audioset_16k") or not any(Path("./audioset_16k").glob("*.wav")):
    try:
        audioset = iter(
            datasets.load_dataset("agkphysics/AudioSet", "balanced", split="train", streaming=True)
                    .cast_column("audio", datasets.Audio(sampling_rate=16000))
        )
        stream_to_wav(audioset, "./audioset_16k", n_clips=N_HOURS_BACKGROUND * 3600 // 10)
    except Exception as e:
        print(f"AudioSet load failed ({type(e).__name__}: {e}). Continuing with FMA only.")

# Free Music Archive — music background (30-second clips)
if not os.path.exists("./fma") or not any(Path("./fma").glob("*.wav")):
    fma = iter(
        datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
                .cast_column("audio", datasets.Audio(sampling_rate=16000))
    )
    stream_to_wav(fma, "./fma", n_clips=N_HOURS_BACKGROUND * 3600 // 30)

# Pre-computed openWakeWord features (training + validation)
feature_base = "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main"
for fname in ("openwakeword_features_ACAV100M_2000_hrs_16bit.npy", "validation_set_features.npy"):
    if not os.path.exists(fname):
        !wget -q {feature_base}/{fname}

# Sanity check — show what actually landed on disk
print("\nBackground data summary:")
for d in ("./mit_rirs", "./audioset_16k", "./fma"):
    n = len(list(Path(d).glob("*.wav"))) if os.path.isdir(d) else 0
    print(f"  {d}: {n} WAVs")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/870 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

AudioSet load failed (TypeError: must be called with a dataclass type or instance). Continuing with FMA only.

Background data summary:
  ./mit_rirs: 270 WAVs
  ./audioset_16k: 0 WAVs
  ./fma: 120 WAVs


## 5. Train and export

Writes a per-run config to `my_model.yaml`, runs the three training stages (`--generate_clips`, `--augment_clips`, `--train_model`), converts the ONNX export to TFLite via `onnx2tf`, then copies both files alongside the notebook as `<wake_word>.{onnx,tflite}`.

`background_paths` is built from "directories that actually contain WAVs" — if AudioSet or FMA fails to download, training continues with whatever you have instead of silently feeding an empty dir. The final download step is also tolerant: if conversion produced only an ONNX (e.g. tensorflow missing), the ONNX still downloads and the tflite step is just skipped with a warning.

In [6]:
# Pick whichever background dirs actually have data — never feed an empty dir to training
candidate_dirs = ["./audioset_16k", "./fma"]
background_paths = [d for d in candidate_dirs if os.path.isdir(d) and any(Path(d).glob("*.wav"))]
if not background_paths:
    raise RuntimeError("No background WAVs found in audioset_16k or fma. Re-run the download cell.")
print(f"Training with background_paths = {background_paths}")

# Build training config from the upstream template
with open("openwakeword/examples/custom_model.yml") as f:
    config = yaml.safe_load(f)

config.update({
    "target_phrase": [WAKE_WORD],
    "model_name": WAKE_WORD.replace(" ", "_"),
    "n_samples": N_SAMPLES,
    "n_samples_val": max(500, N_SAMPLES // 10),
    "steps": N_TRAINING_STEPS,
    "target_accuracy": 0.5,
    "target_recall": 0.25,
    "output_dir": "./my_custom_model",
    "max_negative_weight": FALSE_ACTIVATION_PENALTY,
    "background_paths": background_paths,
    "false_positive_validation_data_path": "validation_set_features.npy",
    "feature_data_files": {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"},
})

with open("my_model.yaml", "w") as f:
    yaml.dump(config, f)

train_cmd = f"{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml"
!{train_cmd} --generate_clips
!{train_cmd} --augment_clips
!{train_cmd} --train_model

# ONNX → TFLite (onnx2tf works on Python 3.11+; the older onnx_tf path is broken there).
# Requires tensorflow at runtime (installed in the setup cell).
name = config["model_name"]
!onnx2tf -i my_custom_model/{name}.onnx -o my_custom_model/ -kat onnx____Flatten_0
if Path(f"my_custom_model/{name}_float32.tflite").exists():
    !mv my_custom_model/{name}_float32.tflite my_custom_model/{name}.tflite

# Copy outputs alongside the notebook as ./<wake_word>.{onnx,tflite} (slug = spaces→underscores).
# In Colab this drops them in /content/ — move them into your repo's models/ dir afterwards.
for ext in ("onnx", "tflite"):
    src = Path(f"my_custom_model/{name}.{ext}")
    if src.exists():
        dst = Path(f"{name}.{ext}")
        shutil.copy(src, dst)
        print(f"Saved {dst}")
    else:
        print(f"WARNING: my_custom_model/{name}.{ext} not produced — check the onnx2tf stderr above.")

# Browser download (Colab only — silently skipped elsewhere). Tolerates a missing tflite.
try:
    from google.colab import files
    for ext in ("onnx", "tflite"):
        path = f"my_custom_model/{name}.{ext}"
        if os.path.exists(path):
            files.download(path)
except ImportError:
    pass

Training with background_paths = ['./fma']
/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py:27: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
INFO:root:##################################################
Generating positive clips for training
##################################################
INFO:root:##################################################
Generating positive clips for testing
##################################################
INFO:root:##################################################
Generating negative clips for training
##################################################
INFO:root:##################################################
Generating negative clips for testing
##################################################
/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py:27: 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>